# Demo: Explainable AI for Product Ranking

End-to-end walkthrough: generate data -> preprocess -> build knowledge graph -> engineer features -> train the hybrid ensemble -> explain a ranking decision -> compute the ECS score -> evaluate.

Run `pip install -r ../requirements.txt` first if you haven't already.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from data.sample_data_generator import generate_dataset
generate_dataset(output_dir='../data/raw', n_users=800, n_products=200, n_interactions=15000, seed=42)

## 1. Load and preprocess

In [ ]:
from preprocessing.data_loader import DatasetLoader
from preprocessing.preprocess import Preprocessor, split_train_val_test

raw = DatasetLoader(dataset='amazon', raw_dir='../data/raw').load()
processed = Preprocessor().run(raw)
train_int, val_int, test_int = split_train_val_test(processed.interactions)
processed.products.head()

## 2. Build the knowledge graph and compute graph features

In [ ]:
from knowledge_graph.graph_builder import KnowledgeGraphBuilder

kg = KnowledgeGraphBuilder()
kg.build(processed.users, processed.products, processed.interactions, processed.reviews)
graph_feats = kg.compute_graph_features().join(kg.compute_graph_embeddings(), how='left').fillna(0.0)
print(f"Graph: {kg.graph.number_of_nodes()} nodes, {kg.graph.number_of_edges()} edges")
graph_feats.head()

## 3. Engineer the full feature matrix

In [ ]:
from feature_engineering.features import FeatureEngineer

fe = FeatureEngineer(graph_features=graph_feats)
full_matrix = fe.build_full_feature_matrix(processed.users, processed.products, processed.interactions, processed.reviews)
print(full_matrix.shape)
full_matrix.head()

## 4. Train the hybrid ranking ensemble

In [ ]:
from ranking.ensemble import HybridRankingEnsemble

keys = train_int[['user_id', 'product_id']].drop_duplicates()
train_matrix = full_matrix.merge(keys, on=['user_id', 'product_id'], how='inner')
feature_cols = [c for c in train_matrix.columns if c not in ['user_id', 'product_id', 'label']]

ensemble = HybridRankingEnsemble()
ensemble.fit(train_matrix[feature_cols], train_matrix['label'], train_matrix['user_id'])
print('Trained models:', list(ensemble.models.keys()))

## 5. Explain a single ranking decision

In [ ]:
from explainability.shap_explainer import ShapRankingExplainer
from explainability.rank_explainer import RankExplainer
from llm.explanation_generator import ExplanationGenerator
from evaluation.ecs import ExplainabilityConfidenceScore

shap_explainer = ShapRankingExplainer(ensemble, train_matrix[feature_cols])
rank_explainer = RankExplainer(shap_explainer, kg)

sample_row = train_matrix[train_matrix['label'] == 1].iloc[[0]]
user_id, product_id = sample_row['user_id'].iloc[0], sample_row['product_id'].iloc[0]
x_row = sample_row[feature_cols]
rank_score = float(ensemble.predict(x_row)[0])

package = rank_explainer.explain(user_id, product_id, x_row, rank_score)
generated = ExplanationGenerator().generate_single_explanation(package)
print(generated.text)
print('Confidence:', generated.confidence_score, '| Passed verification:', generated.verification.passed)

In [ ]:
import numpy as np

all_shap = shap_explainer.explain_instances(x_row).iloc[0].values
top_k_idx = list(range(min(len(package.top_shap_features), len(all_shap))))

ecs = ExplainabilityConfidenceScore().compute(
    all_shap_values=all_shap,
    top_k_indices=top_k_idx,
    graph_path=package.graph_path,
    hallucination_confidence=generated.confidence_score,
    features_run_1=[f['feature'] for f in package.top_shap_features],
)
print('ECS score:', ecs.score)
print(ecs.components)

## 6. Evaluate on the test split

In [ ]:
import pickle, pathlib
pathlib.Path('../results').mkdir(exist_ok=True)
with open('../results/hybrid_ensemble.pkl', 'wb') as fh:
    pickle.dump({'ensemble': ensemble, 'feature_cols': feature_cols}, fh)

from evaluation.evaluate import evaluate_model
report = evaluate_model(ensemble_path='../results/hybrid_ensemble.pkl', output_dir='../results')
report['hybrid_ensemble']